# ForestGuard

## 01 — Extracción de datos

Obtención de datos históricos 2022–2024 y datos 2025 desde fuentes oficiales.

## 1. Configuración del entorno

Se crean las carpetas temporales de trabajo dentro de Google Colab. Los archivos generados se utilizan entre los notebooks durante la misma sesión.

In [ ]:
from pathlib import Path
from datetime import datetime
import requests
import pandas as pd

ROOT = Path("/content")
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"

RAW.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)

print("Carpeta raw:", RAW)
print("Carpeta processed:", PROCESSED)

## 2. Fuentes oficiales

Para los años 2022–2024 se utiliza una capa oficial `eventos_historicos`, que contiene año, tipo de evento, ubicación, hectáreas afectadas y otras variables de impacto.

Para 2025 se mantiene la capa específica de incendios forestales utilizada desde el inicio del proyecto.

In [ ]:
URL_HISTORICO = (
    "https://sgrportal.gestionderiesgos.gob.ec/server/rest/services/"
    "Hosted/eventos_historicos/FeatureServer/0/query"
)

URL_2025 = (
    "https://sgrportal.gestionderiesgos.gob.ec/server/rest/services/"
    "Hosted/incendios_forestales_2025_shp/FeatureServer/0/query"
)

print("Fuente histórica 2022–2024:", URL_HISTORICO)
print("Fuente específica 2025:", URL_2025)

## 3. Función de extracción

La función permite descargar registros de una capa ArcGIS utilizando paginación. Esto evita depender del límite máximo de registros que puede devolver una sola consulta.

In [ ]:
def descargar_arcgis(url, where="1=1", out_fields="*", page_size=2000):
    registros = []
    offset = 0

    while True:
        params = {
            "where": where,
            "outFields": out_fields,
            "returnGeometry": "false",
            "resultOffset": offset,
            "resultRecordCount": page_size,
            "f": "json",
        }

        respuesta = requests.get(url, params=params, timeout=60)
        respuesta.raise_for_status()
        datos = respuesta.json()

        if "error" in datos:
            raise RuntimeError(datos["error"])

        features = datos.get("features", [])

        if not features:
            break

        registros.extend(
            feature.get("attributes", {})
            for feature in features
        )

        if len(features) < page_size:
            break

        offset += page_size

    return pd.DataFrame(registros)

## 4. Extracción histórica 2022–2024

Primero se descargan los eventos correspondientes a los años 2022, 2023 y 2024. Después se filtran únicamente los registros cuyo campo `evento` corresponde a incendios forestales.

El filtro por tipo de evento se realiza en pandas para que el proceso sea fácil de revisar y reproducir.

In [ ]:
# Descargar primero todas las columnas reales de la capa histórica.
# Así evitamos errores si la API cambia o si un nombre no coincide exactamente.
df_historico = descargar_arcgis(
    URL_HISTORICO,
    where="año >= 2022 AND año <= 2024",
    out_fields="*",
)

print("Dimensiones históricas antes del filtro:", df_historico.shape)
print("\nColumnas disponibles en la fuente histórica:")
print(df_historico.columns.tolist())

# Buscar automáticamente la columna que identifica el tipo de evento
candidatas_evento = [
    "evento",
    "categoria_del_evento",
    "tipo_evento",
    "tipo_de_evento",
    "categoria",
]

col_evento = next(
    (c for c in candidatas_evento if c in df_historico.columns),
    None
)

if col_evento is None:
    raise KeyError(
        "No se encontró una columna para identificar el tipo de evento. "
        "Revisa la lista de columnas mostrada arriba."
    )

print("\nColumna utilizada para identificar el evento:", col_evento)

# Ver algunos valores reales antes de filtrar
print("\nTipos de evento encontrados:")
display(
    df_historico[col_evento]
    .astype("string")
    .value_counts(dropna=False)
    .head(20)
    .to_frame("Cantidad")
)

# Filtrar incendios forestales
evento_normalizado = (
    df_historico[col_evento]
    .astype("string")
    .str.strip()
    .str.upper()
)

mascara_incendio = evento_normalizado.str.contains(
    r"INCENDIO.*FOREST",
    regex=True,
    na=False
)

df_historico = df_historico.loc[mascara_incendio].copy()

print("\nRegistros históricos de incendios forestales:", df_historico.shape)

# Verificar los años disponibles
if "año" not in df_historico.columns:
    raise KeyError(
        "La fuente histórica no contiene la columna 'año'. "
        "Revisa las columnas mostradas arriba."
    )

conteo_historico_anio = (
    pd.to_numeric(df_historico["año"], errors="coerce")
    .value_counts()
    .sort_index()
)

display(
    conteo_historico_anio.to_frame("Cantidad de incendios")
)

anios_encontrados = set(
    pd.to_numeric(df_historico["año"], errors="coerce")
    .dropna()
    .astype(int)
    .unique()
)

anios_esperados = {2022, 2023, 2024}
faltantes = anios_esperados - anios_encontrados

if df_historico.empty:
    raise ValueError(
        "La fuente respondió, pero no se encontraron incendios forestales "
        "entre 2022 y 2024."
    )

if faltantes:
    raise ValueError(
        f"Faltan registros de los años: {sorted(faltantes)}. "
        "No continúes con la integración hasta revisar la extracción."
    )

display(df_historico.head())

## 5. Extracción del dataset 2025

Se descarga la capa específica de incendios forestales de 2025. Esta fuente mantiene el conjunto de variables original utilizado en la primera versión de ForestGuard.

In [ ]:
df_2025 = descargar_arcgis(URL_2025)

if df_2025.empty:
    raise ValueError("La fuente 2025 no devolvió registros.")

print("Dataset 2025:", df_2025.shape)

### Vista segura del dataset 2025

Para las evidencias se muestran únicamente variables relacionadas con el evento. Se evita mostrar campos administrativos o de contacto que no son necesarios para el análisis.

In [ ]:
columnas_vista_2025 = [
    "provincia",
    "canton",
    "parroquia",
    "mes",
    "dia",
    "latitud",
    "longitud",
    "causa",
    "hadecobert",
    "nivelinf",
    "region",
    "personasaf",
    "familiasaf",
    "animalesaf",
    "animalesmu",
]

columnas_vista_2025 = [
    c for c in columnas_vista_2025
    if c in df_2025.columns
]

display(df_2025[columnas_vista_2025].head())

## 6. Resumen de la extracción

Se registra la fecha de ejecución, el tamaño de cada fuente y la cantidad de incendios forestales disponibles por año antes de realizar la integración.

In [ ]:
fecha_extraccion = datetime.now().strftime("%Y-%m-%d %H:%M")

conteo_historico = (
    pd.to_numeric(df_historico["año"], errors="coerce")
    .value_counts()
    .sort_index()
)

resumen_extraccion = pd.DataFrame({
    "Fuente": ["Histórico 2022–2024", "Incendios 2025"],
    "Filas": [df_historico.shape[0], df_2025.shape[0]],
    "Columnas": [df_historico.shape[1], df_2025.shape[1]],
})

print("Fecha de extracción:", fecha_extraccion)
display(resumen_extraccion)

print("\nRegistros históricos por año:")
display(conteo_historico.to_frame("Cantidad de registros"))

## 7. Almacenamiento de datos originales

Los dos conjuntos se guardan por separado. La integración se realizará en el siguiente notebook después de homologar nombres y tipos de variables.

In [ ]:
archivo_historico = RAW / "incendios_2022_2024_raw.csv"
archivo_2025 = RAW / "incendios_2025_raw.csv"

df_historico.to_csv(
    archivo_historico,
    index=False,
    encoding="utf-8-sig"
)

df_2025.to_csv(
    archivo_2025,
    index=False,
    encoding="utf-8-sig"
)

print("Histórico guardado en:", archivo_historico)
print("2025 guardado en:", archivo_2025)

## 8. Resultado de la extracción

ForestGuard pasa de trabajar únicamente con registros de 2025 a contar con información de cuatro años: 2022, 2023, 2024 y 2025.

La integración todavía no se realiza en esta etapa porque las fuentes utilizan nombres de columnas diferentes. Esa homologación se desarrolla en el notebook de limpieza y transformación.

## Descarga de archivos para el siguiente módulo

Descarga los dos archivos `raw` generados. Se utilizarán en el módulo 02.

In [ ]:
from google.colab import files

files.download(str(archivo_historico))
files.download(str(archivo_2025))